# Applied ML for Finance — Interview Preparation Notebook

This notebook covers applied machine learning concepts for financial data interviews: feature engineering, time series, model evaluation, validation strategies, and financial context awareness — all explained for beginners.

## Table of Contents

1. **Data & Feature Considerations** — structured, sequential, market-derived data; missing data, drift, non-stationarity
2. **Modeling & Evaluation** — model types for structured/time-dependent data; beyond accuracy; stability, robustness, business metrics
3. **Time Series & Sequential Modeling** — temporal dependencies, lags, autoregression, filtering, forecasting challenges
4. **Feature Engineering for Financial Data** — extracting signals from time-based/market data, domain-driven design
5. **Model Reliability & Validation** — temporal cross-validation, overfitting, performance drift, high-stakes errors
6. **Applied Financial Understanding** — economic indicators, market data as ML inputs, assumptions about data generation

---
## 1. Data & Feature Considerations

**Why it matters:** Financial data is messy, non-stationary (its properties change over time), and often sequential. Working with it requires care that standard tabular data doesn't.

### 🧠 Beginner's Guide

Financial data is different from random datasets you find online. Three things make it special:

1. **It's sequential (time-dependent).** Yesterday's stock price influences today's. You can't shuffle rows like in a normal ML problem — that would create "look-ahead bias" (using future data to predict the past).
2. **It's non-stationary.** The statistical properties of financial data change over time. The mean and variance of a stock's returns in 2020 (COVID crash) look nothing like 2021 (recovery). Models trained in 2020 may fail in 2024.
3. **It's noisy and low signal-to-noise.** Financial markets are efficient — most price movements are random noise. Finding predictable patterns is genuinely hard.

### Key Concepts

| Concept | What It Means | Why It Matters for Finance |
|---|---|---|
| **Missing data** | Gaps in your dataset — holidays, mergers, trading halts | Can't fill with simple mean; need forward-fill, interpolation, or domain-aware imputation |
| **Data drift** | The input distribution P(X) changes over time | A model trained on 2019 data sees different patterns in 2024. Feature distributions (e.g. volatility) shift |
| **Concept drift** | The relationship P(y\|X) changes | What signals a "risky loan" in a low-interest-rate era may not apply when rates rise |
| **Non-stationarity** | Statistical properties (mean, variance) change over time | Returns, volatility, correlations all drift. Rolling windows and differencing help |
| **Survivorship bias** | Only looking at assets that still exist | Ignoring delisted stocks overstates historical returns. Always include dead companies |
| **Look-ahead bias** | Using information not available at prediction time | Using Q4 earnings data to predict Q3 returns — accidentally using future data |
| **Leakage from corporate actions** | Stock splits, dividends, mergers | Price jumps from splits look like signals but aren't. Adjust prices for corporate actions |

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Demonstrate non-stationarity: rolling statistics on simulated price data
np.random.seed(42)
days = 1000
# Simulate a price series with changing volatility (regime shift)
returns = np.random.normal(0.0005, 0.01, days)  # base returns
returns[500:] = np.random.normal(0.0002, 0.03, days - 500)  # higher vol regime
price = 100 * np.exp(returns.cumsum())

df = pd.DataFrame({"price": price})
df["rolling_mean"] = df["price"].rolling(50).mean()
df["rolling_std"] = df["price"].rolling(50).std()

print(f"First 500 days — mean return: {returns[:500].mean():.6f}, "
      f"std: {returns[:500].std():.6f}")
print(f"Last 500 days  — mean return: {returns[500:].mean():.6f}, "
      f"std: {returns[500:].std():.6f}")
print("\nThe distribution changed! A model trained on the first regime "
      "may fail on the second.")
print(f"\nPrice head:\n{df.head()}")
print(f"\nRolling stats head:\n{df[['rolling_mean', 'rolling_std']].head()}")


First 500 days — mean return: 0.000568, std: 0.009803
Last 500 days  — mean return: 0.003455, std: 0.030277

The distribution changed! A model trained on the first regime may fail on the second.

Price head:
        price  rolling_mean  rolling_std
0  100.548211           NaN          NaN
1  100.459502           NaN          NaN
2  101.162848           NaN          NaN
3  102.766751           NaN          NaN
4  102.577677           NaN          NaN

Rolling stats head:
   rolling_mean  rolling_std
0           NaN          NaN
1           NaN          NaN
2           NaN          NaN
3           NaN          NaN
4           NaN          NaN


### Discussion Questions — Data & Features

- **"How do you handle missing data in financial time series?"** → Never use simple mean imputation — that would introduce look-ahead bias. Use forward-fill (carry last observation forward) for prices, or interpolation for intra-day data. For missing trading days (holidays), align to a trading calendar. *Beginner tip: If a stock didn't trade on a holiday, there's no price to fill — just skip that day. If data is missing for a few minutes, carry forward the last known price.*
- **"What's non-stationarity and why does it matter for ML?"** → Financial data changes character over time (volatility clusters, regime shifts, drift). Models trained on stationary assumptions will fail when the regime changes. Mitigations: differencing (predict changes not levels), rolling windows, regime-switching models. *Beginner tip: Imagine training a weather model on summer data and using it in winter — it'll be wrong. Financial data is like that but on steroids.*
- **"How do you detect data drift in production?"** → Track feature distributions with statistical tests (Kolmogorov-Smirnov test, Population Stability Index). Monitor prediction distributions. Set alert thresholds. Tools: Evidently, NannyML, WhyLabs. *Beginner tip: If the average volatility of your features suddenly doubles, something has changed — your model's predictions are less reliable.*

---
## 2. Modeling & Evaluation

**Why it matters:** Choosing the right model and evaluation strategy for financial data requires going beyond off-the-shelf metrics.

### 🧠 Beginner's Guide

Financial modeling problems usually fall into a few categories:

- **Classification:** Will this loan default? Is this transaction fraudulent? Will the price go up or down?
- **Regression:** What will the stock price be? What's the expected return? What's the volatility?
- **Time series forecasting:** What will sales be next quarter? What's the GDP growth rate?

**Beyond accuracy:** In finance, a model that's 95% accurate could be terrible if the 5% of mistakes are catastrophic (e.g. missing a market crash). You need to evaluate:
- **Stability:** Does the model perform consistently across different market regimes?
- **Robustness:** How much does performance drop when input distributions shift?
- **Business metrics:** Does the model make money? Does it reduce risk? Accuracy alone doesn't answer these.

### Model Types for Financial Data

| Model | Best For | Why It Works | Warning |
|---|---|---|---|
| **Linear Regression** | Simple factor models, beta estimation | Interpretable, fast | Assumes linearity — markets are rarely linear |
| **Logistic Regression** | Credit scoring, default prediction | Well-calibrated probabilities | Linear boundary may miss complex interactions |
| **Decision Tree / RF** | Feature importance, risk segmentation | Handles non-linearity, robust to outliers | Can overfit to noise in financial data |
| **XGBoost / LightGBM** | Tabular financial data, fraud detection | State-of-the-art for structured data, handles missing values | Needs careful tuning to avoid overfitting, many hyperparameters |
| **LSTM / GRU** | Sequential patterns, volatility forecasting | Captures long-term dependencies in sequences | Needs lots of data, prone to overfitting, black box |
| **Transformer / Attention** | Cross-asset patterns, limit order book | Best for capturing complex dependencies | Very data-hungry, computationally expensive |
| **ARIMA / GARCH** | Pure time series (volatility, returns) | Statistical foundation, well-understood | Assumes stationarity, limited feature flexibility |
| **Kalman Filter** | Real-time tracking, dynamic systems | Handles non-stationarity naturally | Linear Gaussian assumption, complex to tune |

In [2]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Simple demo: predict next-day return using lagged features
np.random.seed(42)
n = 500
df_demo = pd.DataFrame({
    "return_t": np.random.normal(0.001, 0.02, n),
    "volume_t": np.random.randint(1_000_000, 10_000_000, n),
    "volatility_t": np.abs(np.random.normal(0.01, 0.005, n)),
})
df_demo["return_t+1"] = (  # target: next day return with some signal
    0.1 * df_demo["return_t"]
    - 0.05 * df_demo["volatility_t"]
    + np.random.normal(0, 0.02, n)
)

X = df_demo[["return_t", "volume_t", "volatility_t"]]
y = df_demo["return_t+1"]

# Train/test split (time-aware: no shuffling!)
split = int(0.8 * n)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print(f"MAE:  {mean_absolute_error(y_test, y_pred):.6f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.6f}")
print(f"R²:   {rf.score(X_test, y_test):.6f}")
print("\n(Benchmark: predicting 0 always would give RMSE ≈ "
      f"{np.sqrt((y_test ** 2).mean()):.6f})")


MAE:  0.019649
RMSE: 0.024369
R²:   -0.045214

(Benchmark: predicting 0 always would give RMSE ≈ 0.023838)


### Discussion Questions — Modeling & Evaluation

- **"How do you evaluate a model for a trading strategy beyond accuracy?"** → Use financial metrics: Sharpe ratio (return per unit risk), maximum drawdown (worst loss), win rate, profit factor. A model with 55% accuracy but huge wins and small losses can be great. A model with 90% accuracy but catastrophic losses is terrible. *Beginner tip: In finance, it's not how often you're right — it's how much you make when you're right vs how much you lose when you're wrong.*
- **"What's the Sharpe ratio?"** → (Portfolio return - Risk-free rate) / Standard deviation of returns. Measures risk-adjusted return. Sharpe > 1 is good, > 2 is great, > 3 is suspicious. *Beginner tip: It answers: "Is the extra return worth the extra risk?" A roller coaster that goes up 30% and down 20% has a lower Sharpe than a slow climb from 1% to 2%.*
- **"How do you handle model decay in production?"** → Monitor prediction drift concept drift. Set up automated retraining pipelines triggered by performance thresholds (e.g. if AUC drops by 0.02, retrain). Shadow-deploy challenger models alongside the champion. *Beginner tip: Models are like milk — they have an expiration date. You need to check regularly whether they're still fresh.*

---
## 3. Time Series & Sequential Modeling

**Why it matters:** Most financial data is sequential — yesterday affects today. Standard ML assumes independence, which breaks down.

### 🧠 Beginner's Guide

Time series is different from regular ML because **order matters**. You can't shuffle data. Tomorrow depends on today.

**Key ideas:**

- **Autocorrelation:** Today's return is correlated with yesterday's return (and maybe the day before). If a stock went up yesterday, it might go up again today (momentum) or down (mean reversion).
- **Stationarity:** A time series is stationary if its mean and variance don't change over time. Most financial data is NOT stationary. We make it stationary through **differencing** — instead of predicting the price, predict the *change* in price.
- **Lag features:** Use yesterday's value as a feature to predict today's. `price_t-1`, `price_t-2`, etc. This is the simplest way to capture sequential patterns.
- **Rolling windows:** Calculate moving averages, rolling standard deviations, rolling correlations. These capture recent trends and volatility.

### Common Approaches

| Approach | Description | When to Use |
|---|---|---|
| **Lag features + tree model** | Add `y_{t-1}, y_{t-2}` as features to RF/XGBoost | Simple, flexible, works well with tabular mindset |
| **ARIMA** | AutoRegressive Integrated Moving Average — statistical model for univariate time series | Baseline for univariate forecasting, well-understood |
| **Exponential Smoothing** | Weighted average of past observations, more weight to recent | Smooth, trend + seasonality decomposition |
| **LSTM / GRU** | Recurrent neural networks that learn temporal dependencies | Complex patterns, enough data, willing to tune |
| **Transformer (Time Series)** | Attention-based models (Informer, Autoformer) | State-of-the-art for long sequences |
| **Kalman Filter** | Recursive filter that estimates hidden states from noisy observations | Real-time tracking, dynamic systems |
| **GARCH** | Models volatility as time-dependent | Volatility forecasting, risk management |

### Forecasting Challenges

- **The random walk problem:** Many financial series (especially stock prices) follow a random walk — the best prediction for tomorrow is today's price. Be very sceptical of models claiming to predict prices accurately.
- **Non-stationarity:** A model trained on last year's data may not work this year. Use rolling windows and retrain frequently.
- **Low signal-to-noise:** Financial data is incredibly noisy. A strategy with a Sharpe of 1.0 means you're making about 1 unit of return per 1 unit of risk — that's considered good in finance but would be terrible in physics.

In [3]:
# Demo: lag features and autocorrelation
df_ts = pd.DataFrame({"price": price})  # from earlier simulation
df_ts["return"] = df_ts["price"].pct_change()

# Create lag features
df_ts["lag_1"] = df_ts["return"].shift(1)
df_ts["lag_2"] = df_ts["return"].shift(2)
df_ts["lag_3"] = df_ts["return"].shift(3)
df_ts["ma_5"] = df_ts["return"].rolling(5).mean()
df_ts["vol_5"] = df_ts["return"].rolling(5).std()

# Autocorrelation of returns
acf_1 = df_ts["return"].autocorr(lag=1)
acf_2 = df_ts["return"].autocorr(lag=2)
acf_5 = df_ts["return"].autocorr(lag=5)

print(f"Autocorrelation at lag 1: {acf_1:.4f}")
print(f"Autocorrelation at lag 2: {acf_2:.4f}")
print(f"Autocorrelation at lag 5: {acf_5:.4f}")
print(f"\n(Returns have low autocorrelation — consistent with efficient markets)")
print(f"Feature matrix shape: {df_ts.dropna().shape}")
print(f"\nTail of feature matrix:\n{df_ts[['return','lag_1','lag_2','ma_5','vol_5']].tail()}")


Autocorrelation at lag 1: -0.0184
Autocorrelation at lag 2: 0.0185
Autocorrelation at lag 5: -0.0086

(Returns have low autocorrelation — consistent with efficient markets)
Feature matrix shape: (995, 7)

Tail of feature matrix:
       return     lag_1     lag_2      ma_5     vol_5
995  0.062243  0.047560  0.005753  0.035790  0.040116
996  0.064010  0.062243  0.047560  0.032596  0.036157
997  0.037123  0.064010  0.062243  0.043338  0.023737
998  0.031405  0.037123  0.064010  0.048468  0.014594
999  0.018138  0.031405  0.037123  0.042584  0.019987


### Discussion Questions — Time Series

- **"How do you make a time series stationary?"** → Differencing (subtract previous value: `y_t - y_{t-1}`), log transforms (stabilise variance), seasonal differencing (for weekly/monthly patterns). Test with Augmented Dickey-Fuller (ADF) test for stationarity. *Beginner tip: If the price goes from 100 → 110 → 90, the series isn't stationary (the level keeps changing). But the changes: +10, -20 may be stationary.*
- **"What's the efficient market hypothesis and why does it matter for ML?"** → EMH says asset prices reflect all available information, so you can't consistently predict future prices. If you believe in strong-form EMH, ML for price prediction is futile. Most practitioners believe markets are *mostly* efficient — there are small, temporary inefficiencies ML can exploit. *Beginner tip: The market is like a very competitive exam — everyone knows the same material. To outperform, you need an edge that others don't have.*
- **"How do you handle multiple time series (e.g. 1000 stocks)?"** → Panel data approach: use entity embeddings to capture stock-specific effects, share the same model across all stocks (cross-sectional learning), or use hierarchical forecasting. *Beginner tip: Instead of training 1000 separate models, train one model that takes a "stock ID" as input and learns which patterns are universal vs stock-specific.*

---
## 4. Feature Engineering for Financial Data

**Why it matters:** In finance, feature engineering is where the real value is created. Raw prices are useless — the signal is in derived features.

### 🧠 Beginner's Guide

Financial data comes in many forms, but the core challenge is turning raw market data into predictive signals.

**Common financial data types:**
- **Price/return data:** Open, high, low, close, volume (OHLCV) for each instrument
- **Fundamental data:** Earnings, P/E ratio, debt/equity, revenue growth (quarterly)
- **Alternative data:** News sentiment, social media, satellite images, credit card transactions
- **Macroeconomic data:** GDP, inflation (CPI), interest rates, unemployment

**Feature families for financial ML:**

| Category | Examples | Why They Work |
|---|---|---|
| **Price-based** | Returns (log, simple), moving averages, % change | Capture trend and momentum |
| **Volatility** | Rolling std dev, ATR (average true range), GARCH estimates | Risk changes over time |
| **Momentum** | RSI (relative strength), MACD, rate of change | Trend-following signals |
| **Volume-based** | Volume change, OBV (on-balance volume), volume-weighted avg | Confirm price moves |
| **Cross-sectional** | Rank within sector, percentile of valuation metric | Relative value signals |
| **Technical indicators** | Bollinger Bands, Support/Resistance levels | Price level context |
| **Fundamental ratios** | P/E, P/B, EV/EBITDA, dividend yield | Company valuation |
| **Calendar/datetime** | Day of week, month, quarter, is_month_end, days_to_earnings | Seasonal patterns |
| **Rolling correlations** | Correlation with market, sector, peer group | Exposure measurement |
| **Lagged interactions** | Return × volume, volatility × volume | Non-linear relationships |

### The Danger of Overfitting in Finance

Financial data has a terrible **signal-to-noise ratio** (maybe 5-10% signal, 90-95% noise). This makes overfitting extremely easy. Warning signs:
- Great backtest performance but poor live trading
- Performance drops significantly when you change the time period
- Features that "make sense" in hindsight but have no economic rationale

In [4]:
# Feature engineering demo on simulated OHLCV data
np.random.seed(42)
n = 200
df_ohlcv = pd.DataFrame({
    "close": 100 + np.cumsum(np.random.normal(0, 1, n)),
    "high": 0,
    "low": 0,
    "volume": np.random.randint(1_000_000, 10_000_000, n),
})
df_ohlcv["high"] = df_ohlcv["close"] * (1 + np.abs(np.random.normal(0, 0.01, n)))
df_ohlcv["low"] = df_ohlcv["close"] * (1 - np.abs(np.random.normal(0, 0.01, n)))

# Feature engineering
df_ohlcv["return"] = df_ohlcv["close"].pct_change()
df_ohlcv["log_return"] = np.log(df_ohlcv["close"] / df_ohlcv["close"].shift(1))
df_ohlcv["range"] = df_ohlcv["high"] - df_ohlcv["low"]
df_ohlcv["ma_10"] = df_ohlcv["close"].rolling(10).mean()
df_ohlcv["vol_10"] = df_ohlcv["return"].rolling(10).std()
df_ohlcv["volume_change"] = df_ohlcv["volume"].pct_change()
# RSI (Relative Strength Index) — simplified
gain = df_ohlcv["return"].clip(lower=0).rolling(14).mean()
loss = (-df_ohlcv["return"].clip(upper=0)).rolling(14).mean()
df_ohlcv["rsi_14"] = 100 - 100 / (1 + gain / (loss + 1e-10))
# Target: next day return direction
df_ohlcv["target_up"] = (df_ohlcv["close"].shift(-1) > df_ohlcv["close"]).astype(int)

print("Feature-engineered DataFrame:")
print(df_ohlcv[["close", "return", "range", "ma_10", "vol_10", "rsi_14", "target_up"]].dropna().head(10))
print(f"\nTotal rows with features: {df_ohlcv.dropna().shape[0]}")


Feature-engineered DataFrame:
         close    return     range       ma_10    vol_10     rsi_14  target_up
14  100.155228 -0.016931  1.893668  103.192461  0.010414  48.708780          0
15   99.592940 -0.005614  0.978697  102.945668  0.010475  46.847703          0
16   98.580109 -0.010170  0.553757  102.439669  0.008582  39.699819          1
17   98.894357  0.003188  0.855113  101.888352  0.007990  32.721705          0
18   97.986332 -0.009182  0.824216  101.293181  0.008070  30.663225          0
19   96.574029 -0.014413  0.757706  100.502522  0.007441  27.610270          1
20   98.039678  0.015176  1.554849   99.904771  0.010419  27.433395          0
21   97.813901 -0.002303  1.281145   99.331015  0.010473  22.273261          1
22   97.881429  0.000690  1.489813   98.739815  0.010346  23.639639          0
23   96.456681 -0.014556  2.579626   98.197468  0.009882  17.547255          0

Total rows with features: 186


### Discussion Questions — Feature Engineering

- **"How do you avoid overfitting in financial feature engineering?"** → Use economic rationale (every feature should have a reason for existing), out-of-time validation (test on a later time period), feature stability analysis (does the feature's predictive power persist?), and dimensionality reduction (PCA, feature selection). *Beginner tip: If you create 100 features by randomly combining price data, some will look great by pure chance. That's overfitting. Every feature needs a story: "This feature works because momentum persists for 10 days."*
- **"What technical indicators would you use for short-term vs long-term trading?"** → Short-term (minutes/hours): order book imbalance, bid-ask spread, volume profiles, micro-price. Long-term (days/months): moving averages, RSI, MACD, fundamental ratios. *Beginner tip: Short-term models focus on liquidity and order flow. Long-term models focus on value and trends.*
- **"How do you handle corporate actions in feature engineering?"** → Always adjust prices for splits, dividends, and mergers. Unadjusted prices create false signals (a 2-for-1 split looks like a 50% price drop). Use adjusted close prices from data vendors. *Beginner tip: When a stock splits 2-for-1, the price halves but nothing has changed. If you don't adjust, your model thinks there was a crash — that's a false signal.*

---
## 5. Model Reliability & Validation

**Why it matters:** Financial models that work in backtests often fail in production. Rigorous validation is essential.

### 🧠 Beginner's Guide

In finance, you can't just split data randomly into train/test. Time matters. Validation for financial models has unique requirements:

1. **No look-ahead bias:** When making a prediction for day T, you can only use information available *before* day T.
2. **No leakage:** Features from day T shouldn't use data from day T+1.
3. **Regime awareness:** Performance in bull markets may differ from bear markets.

### Temporal Cross-Validation Strategies

| Method | How It Works | Best For |
|---|---|---|
| **Expanding window** | Train on [1..t], test on t+1. Grow training set over time | When more data always helps |
| **Rolling window** | Train on [t-window..t], test on t+1. Fixed-size training window | When old data becomes irrelevant |
| **Purged k-fold** | Standard k-fold but remove data near the test fold boundary | When you have enough data and need robust estimates |
| **Combinatorial CV** | All possible train/test splits preserving time order | Hyperparameter tuning |
| **Walk-forward** | Sequential train/test windows, like cross-validation for time series | Most common in quant finance |

### Overfitting in Finance

Financial data has more noise than signal, making overfitting the #1 danger. Common overfitting traps:

| Trap | What It Looks Like | Prevention |
|---|---|---|
| **Backtest overfitting** | Great performance in backtest, terrible live | Out-of-sample testing, walk-forward analysis |
| **Data snooping** | Testing too many ideas on the same data | Hold out a final test set, adjust p-values for multiple tests |
| **Selection bias** | Choosing assets that performed well historically | Include delisted assets, random asset selection |
| **Survivorship bias** | Ignoring failed companies/funds | Include delisted/dead assets in your dataset |
| **Over-optimisation** | Parameters tuned perfectly to historical data | Simpler models, parameter stability analysis |

In [5]:
# Demo: walk-forward validation (expanding window)

from sklearn.metrics import accuracy_score

def walk_forward_validate(X, y, model_class, window_start=100, **model_kwargs):
    """Simple expanding-window walk-forward validation."""
    n = len(X)
    predictions = []
    actuals = []

    for t in range(window_start, n):
        # Train on data up to t-1
        model = model_class(**model_kwargs)
        model.fit(X.iloc[:t], y.iloc[:t])
        # Predict t
        pred = model.predict(X.iloc[t:t+1])[0]
        predictions.append(pred)
        actuals.append(y.iloc[t])

    return np.array(predictions), np.array(actuals)

# Use OHLCV data from earlier for binary classification
df_wf = df_ohlcv.dropna().copy()
feature_cols = ["return", "range", "volume_change", "rsi_14", "vol_10"]
X_wf = df_wf[feature_cols]
y_wf = df_wf["target_up"]

from sklearn.ensemble import RandomForestClassifier
preds, actuals = walk_forward_validate(X_wf, y_wf, RandomForestClassifier,
                                        window_start=50, n_estimators=50, random_state=42)

acc = accuracy_score(actuals, preds)
print(f"Walk-forward accuracy: {acc:.3f}")
print(f"Total predictions: {len(preds)}")
print(f"Benchmark (always predict previous direction): "
      f"{max((actuals == 1).mean(), (actuals == 0).mean()):.3f}")


Walk-forward accuracy: 0.500
Total predictions: 136
Benchmark (always predict previous direction): 0.537


### Discussion Questions — Reliability & Validation

- **"Why is standard k-fold cross-validation dangerous for financial data?"** → Standard k-fold shuffles data randomly. In financial data, this creates look-ahead bias — the model trains on future data and predicts past data. Performance will be artificially high. Always use time-aware cross-validation (walk-forward, expanding window). *Beginner tip: Imagine training a model on 2021 data and testing on 2020 — your model already "knows" what happened. That's cheating. Time must flow forward.*
- **"How do you measure if your model is actually adding value vs random?"** → Compare against naive benchmarks: predicting the mean, predicting last period's value, or a simple heuristic (like buying and holding). Use statistical tests (Diebold-Mariano for forecasts). Track Sharpe ratio, not just accuracy. *Beginner tip: If your complex ML model barely beats "predict tomorrow is the same as today," it's not useful. A model must earn its complexity by clearly outperforming simple baselines.*
- **"How would you detect and handle model degradation in production?"** → Monitor prediction distributions, feature distributions, and actual outcomes (when available). Set up automated alerts for drift. Implement shadow deployment (a challenger model runs alongside the champion). Schedule retraining. *Beginner tip: If your model's average prediction suddenly shifts from 0.3 to 0.6, something changed — either the market regime shifted or your model is broken. Investigate before blindly retraining.*

---
## 6. Applied Financial Understanding

**Why it matters:** You don't need to be a quant, but you need to reason about how financial data behaves and how economic context affects ML models.

### 🧠 Beginner's Guide

You don't need to be a finance expert, but interviewers want to see that you understand the *nature* of financial data and can reason about it.

### Key Economic Concepts for ML Practitioners

| Concept | What It Is | Why It Matters for ML |
|---|---|---|
| **Risk-free rate** | Return on "safe" assets (government bonds) | The baseline return — your model needs to beat this. If the risk-free rate is 5%, a model returning 6% with high risk is unattractive |
| **Volatility** | Standard deviation of returns | Key input for risk models. Volatility clusters — calm markets are followed by calm, volatile by volatile |
| **Correlation** | How assets move together | Correlations change in crises (everything goes down together). Dynamic correlation is a modeling challenge |
| **Liquidity** | How easily an asset can be traded | Models predicting price moves in illiquid assets may not be actionable — you can't trade without moving the price |
| **Market regimes** | Different market environments (bull/bear/crash/calm) | A model trained in one regime may fail in another. Regime detection is often necessary |
| **Mean reversion** | Prices tend to return to some average over time | Basis for many trading strategies. But the "average" itself changes over time |
| **Momentum** | Assets that performed well continue to perform well | Another common signal. Works over certain time horizons (6-12 months) |
| **Seasonality** | Calendar-based patterns (January effect, Monday effect) | Weak signals but can add marginal predictive power |
| **Arbitrage** | Exploiting price differences across markets | Rare in modern markets — arbitrage opportunities disappear quickly as traders compete |

### Common Financial Data Sources for ML

| Source | Examples | Challenges |
|---|---|---|
| **Market data** | OHLCV, order book, trade data | High volume, noisy, non-stationary |
| **Fundamental data** | Balance sheets, income statements | Low frequency (quarterly), restatements, reporting delays |
| **Alternative data** | Satellite imagery, credit card transactions, web scraping | Expensive, messy, regulatory concerns |
| **Macroeconomic** | GDP, CPI, unemployment, PMI | Low frequency, revisions, lagged releases |
| **Sentiment** | News, social media, analyst reports | Hard to quantify, unstructured, fast-decaying value |

### The Key Principle: Economic Rationale

The best financial ML features have an **economic rationale** — a story about *why* they should work. A feature that "worked" in backtest but has no economic basis is almost certainly overfitting. Always ask: "Is there a reason this should predict future returns?"

In [6]:
# Simple demo: economic rationale — momentum vs mean reversion

np.random.seed(42)
days = 500
# Simulate a price with short-term mean reversion + long-term trend
noise = np.random.normal(0, 1, days)
trend = np.linspace(0, 10, days)  # gentle upward trend
price_sim = 100 + trend + noise

df_econ = pd.DataFrame({"price": price_sim})
df_econ["return"] = df_econ["price"].pct_change()
df_econ["ma_20"] = df_econ["price"].rolling(20).mean()
df_econ["ma_50"] = df_econ["price"].rolling(50).mean()
df_econ["momentum"] = df_econ["ma_20"] / df_econ["ma_50"] - 1  # > 0 = short-term > long-term
df_econ["distance"] = df_econ["price"] / df_econ["ma_50"] - 1  # how far from 50-day avg

print("Economic rationale:")
print("- momentum > 0: short-term trend is above long-term trend (momentum signal)")
print("- distance > 0: price is above its 50-day average (could mean-revert)")
print()
print(df_econ[["price", "momentum", "distance"]].dropna().tail(10))
print()
print(f"Correlation between momentum and next-day return: "
      f"{df_econ['momentum'].shift(1).corr(df_econ['return']):.4f}")
print("(Low correlation = market is efficient. Any edge is tiny.)")


Economic rationale:
- momentum > 0: short-term trend is above long-term trend (momentum signal)
- distance > 0: price is above its 50-day average (could mean-revert)

          price  momentum  distance
490  109.811667  0.004201  0.005781
491  111.319623  0.005653  0.019123
492  109.937088  0.005986  0.005886
493  109.018475  0.005108 -0.002561
494  111.422924  0.004532  0.018724
495  110.458750  0.004915  0.009748
496  108.902634  0.004434 -0.004404
497  109.769581  0.004067  0.003201
498  109.104342  0.002326 -0.002996
499  108.617200  0.001241 -0.007570

Correlation between momentum and next-day return: -0.1387
(Low correlation = market is efficient. Any edge is tiny.)


### Discussion Questions — Applied Financial Understanding

- **"How does the efficient market hypothesis affect your approach to feature engineering?"** → If markets are mostly efficient, most features will be useless. Focus on capturing market frictions (liquidity, transaction costs), behavioural biases (overreaction, underreaction), or structural constraints (regulations, institutional mandates). Don't expect to find "free lunch" features. *Beginner tip: If it were easy to predict stock prices, everyone would do it. Your edge must come from better data, better processing, or a unique insight that others don't have.*
- **"What's the difference between alpha and beta?"** → *Beta* is exposure to the overall market (systematic risk). *Alpha* is excess return beyond what beta explains — the value your model adds. An alpha of 0 means you're just tracking the market. *Beginner tip: Beta is like a boat rising with the tide. Alpha is like having a motor that makes you go faster than the tide. Your ML model should generate alpha.*
- **"How would you handle regime changes in a trading model?"** → Use regime detection models (hidden Markov models, clustering on volatility/returns). Train separate models for different regimes. Use adaptive learning (models that update online). Monitor for regime shifts and trigger retraining. *Beginner tip: The 2008 financial crisis was a regime change — models trained on 2003-2007 data failed completely. Your model needs to either detect regime shifts or be robust enough to handle different market environments.*
- **"How do you think about transaction costs when evaluating a model?"** → Transaction costs (commissions, bid-ask spread, market impact) can turn a profitable strategy into a losing one. Always include realistic cost assumptions in your evaluation. High-turnover strategies (many trades) are especially sensitive to costs. *Beginner tip: A strategy that makes $0.01 per trade but costs $0.02 in fees is losing money. Always model costs before concluding a strategy works.*